# EWS 2D MAE pretraining (standalone from repo `pretrain/`)

Same **text list → load by path** style as AR-SSL4M; does **not** modify files under root `pretrain/`.

## Data

- **`float32` `.npy`** crops, shape **`(350, 350, 3)`**, under **`EWS/data/npy/`**（首格 `NPY_DIR`；若相对路径不存在则回退 `D:\Cursor\UNSW-COMP-9517\EWS\data\npy`）。
- Cell below builds **`EWS/data/pretrain/patch_lists/all_patches.txt`** (one absolute path per line).

## Model

- **2D MAE**: `Conv2d` patch embed + `TransformerEncoder`, **RGB**, **`patch_size=5`** (350/5 = 70×70 patches)。训练格使用 **`batch_size=1`**。
- Code in **`EWS/pretrain/`** (`dataset_npy.py`, `mae2d.py`, `train_mae2d.py`).

## Dependencies

- `torch` (**CUDA build**; **RTX 50 / Blackwell needs cu128-class wheels**, cu124 may raise `no kernel image`), `tqdm`.
- Training **defaults to GPU**; pass `device="cpu"` only for debugging.


In [1]:
from pathlib import Path


def find_repo(start: Path) -> Path:
    for p in [start.resolve(), *start.resolve().parents]:
        if (p / "EWS" / "pretrain" / "train_mae2d.py").is_file():
            return p
    raise FileNotFoundError("Open this notebook inside the AR-SSL4M repo.")


REPO = find_repo(Path.cwd())
NPY_DIR = REPO / "EWS" / "data" / "npy"
_NPY_FALLBACK = Path(r"D:\Cursor\UNSW-COMP-9517\EWS\data\npy")
if not NPY_DIR.is_dir() and _NPY_FALLBACK.is_dir():
    NPY_DIR = _NPY_FALLBACK
LIST_DIR = REPO / "EWS" / "data" / "pretrain" / "patch_lists"
LIST_FILE = LIST_DIR / "all_patches.txt"
CKPT_DIR = REPO / "EWS" / "data" / "pretrain" / "checkpoints"

if not NPY_DIR.is_dir():
    raise FileNotFoundError(f"Prepare this directory first: {NPY_DIR}")

print("NPY_DIR:", NPY_DIR.resolve())

LIST_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)


NPY_DIR: D:\Cursor\UNSW-COMP-9517\EWS\data\npy


In [2]:
count_1 = 0
count_2 = 0

paths = sorted(NPY_DIR.glob("*.npy"))
with open(LIST_FILE, "w", encoding="utf-8") as f:
    for p in paths:
        f.write(str(p.resolve()) + "\n")
        count_1 += 1
        count_2 += 1
        if count_2 >= 100:
            print(count_1)
            count_2 = 0

print("list_path:", LIST_FILE)
print("num_samples:", len(paths))


100
200
300
400
500
600
700
800
900
1000
1100
1200
1300
1400
1500
1600
1700
1800
1900
2000
2100
2200
2300
2400
2500
2600
2700
2800
2900
3000
3100
3200
3300
3400
3500
3600
3700
3800
3900
4000
4100
4200
4300
4400
4500
4600
4700
4800
4900
5000
5100
5200
5300
5400
5500
5600
5700
5800
5900
6000
6100
6200
6300
6400
6500
6600
6700
6800
6900
7000
7100
7200
7300
7400
7500
7600
7700
7800
7900
8000
8100
8200
8300
8400
8500
8600
8700
8800
8900
9000
9100
9200
9300
9400
9500
9600
9700
9800
9900
10000
10100
10200
10300
10400
10500
10600
10700
10800
10900
11000
11100
11200
11300
11400
11500
11600
11700
11800
11900
12000
12100
12200
12300
12400
12500
12600
12700
list_path: D:\Cursor\UNSW-COMP-9517\EWS\data\pretrain\patch_lists\all_patches.txt
num_samples: 12714


In [3]:
# Blackwell (RTX 5090, sm_120): need CUDA 12.8+ wheels; cu124 may hit no kernel image.
# Default: nightly cu128 + --pre. Override with TORCH_INDEX_URL / TORCH_PIP_PRE=0; see install.py.
import os
import subprocess
import sys

idx = os.environ.get("TORCH_INDEX_URL", "https://download.pytorch.org/whl/nightly/cu128")
extra = ["--pre"] if os.environ.get("TORCH_PIP_PRE", "1") not in ("0", "false", "False") else []
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "--upgrade", "--force-reinstall", *extra, "torch", "--index-url", idx]
)


0

In [4]:
# After pip install, restart kernel before this cell (old torch may still be loaded).
import torch

v = torch.__version__
print("torch:", v)
print("torch.version.cuda:", torch.version.cuda)

if "+cpu" in v:
    raise RuntimeError(
        "Still CPU-only PyTorch (+cpu in version). Run install cell with --force-reinstall, then restart kernel."
    )
if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA not available. Restart kernel after install, or use cu128-class wheels for Blackwell."
    )
cap = torch.cuda.get_device_capability(0)
print("compute capability:", cap, "(12,0)=Blackwell sm_120")
print("GPU:", torch.cuda.get_device_name(0))
try:
    a = torch.randn(256, 256, device="cuda")
    b = torch.randn(256, 256, device="cuda")
    _ = a @ b
    torch.cuda.synchronize()
except RuntimeError as e:
    if "kernel image" in str(e).lower():
        raise RuntimeError(
            "GPU matmul failed (often RTX 50 + cu124/cu121 wheel). Use nightly/cu128 install cell, restart kernel."
        ) from e
    raise


torch: 2.12.0.dev20260408+cu128
torch.version.cuda: 12.8
compute capability: (12, 0) (12,0)=Blackwell sm_120
GPU: NVIDIA GeForce RTX 5090 D


In [5]:
import sys

EWS_PRE = REPO / "EWS" / "pretrain"
sys.path.insert(0, str(EWS_PRE))

from dataset_npy import NpyPatchListDataset

ds = NpyPatchListDataset(LIST_FILE, split="train")
print("train:", len(ds), "val:", len(NpyPatchListDataset(LIST_FILE, split="val")))
x = ds[0]
print("single sample shape (3,H,W):", x.shape)


train: 12688 val: 26
single sample shape (3,H,W): torch.Size([3, 350, 350])


In [6]:
import sys

sys.path.insert(0, str(REPO / "EWS" / "pretrain"))
from train_mae2d import run_training

run_training(
    list_path=str(LIST_FILE.resolve()),
    ckpt_dir=str(CKPT_DIR.resolve()),
    img_size=350,
    patch_size=5,
    epochs=5,
    batch_size=1,
    num_workers=2,
    val_every=500,
    mask_ratio=0.5,
)


Training device: cuda (NVIDIA GeForce RTX 5090 D)


D:\Cursor\UNSW-COMP-9517\EWS\pretrain\mae2d.py:57: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=depth)
D:\Cursor\UNSW-COMP-9517\EWS\pretrain\mae2d.py:71: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.decoder = nn.TransformerEncoder(dec_layer, num_layers=decoder_depth)


epoch 1/5  train_loss=0.004895  val_loss=0.004431


epoch 2/5  train_loss=0.004271  val_loss=0.004834


epoch 3/5  train_loss=0.004208  val_loss=0.004688


epoch 4/5  train_loss=0.004166  val_loss=0.004581


epoch 5/5  train_loss=0.004133  val_loss=0.004583
